### Load the shared setup context

**Purpose:** Initialize the catalog, schema, and runtime variables used to validate dashboard sources.

**Inputs:** Demo2 Olist setup parameters.

**Outputs:** The target context used by all dashboard validation queries.

**Why it matters:** Dashboard checks must validate the same development tables that the Job will refresh.

In [0]:
%run ./00_setup

### Load the dashboard validation context

**Purpose:** Establish the shared Demo2 Olist catalog and schema before checking dashboard sources.

**Inputs:** Setup parameters and the current development namespace.

**Outputs:** Runtime variables used by dashboard validation queries.

**Why it matters:** The dashboard must be validated against the same Gold outputs produced by the Job.

In [0]:
# Cell 2 — Select the project namespace and verify the dashboard environment.
spark.sql(f"USE CATALOG `{catalog}`").collect()
spark.sql(f"USE SCHEMA `{schema}`").collect()

display(
    spark.sql("""
        SELECT
            current_catalog() AS current_catalog,
            current_schema() AS current_schema
    """)
)

### Select the project namespace

**Purpose:** Set and display the active catalog and schema for dashboard checks.

**Inputs:** `catalog` and `schema` from setup.

**Outputs:** An explicit SQL context for source-table validation.

**Why it matters:** Namespace verification prevents a dashboard from being certified against the wrong environment.

In [0]:
# Cell 3 — Create the governed dashboard view from the Gold star schema.
spark.sql(f"""
    CREATE OR REPLACE VIEW
        {catalog}.{schema}.secure_gold_order_items_dashboard
    AS
    SELECT
        f.order_item_sk,
        f.order_id,
        f.order_item_id,
        f.order_status,
        f.order_purchase_timestamp,

        f.customer_sk,
        c.customer_id,
        c.customer_unique_id,
        c.customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,

        f.product_sk,
        p.product_id,
        p.product_category_name,
        COALESCE(
            p.product_category_name_english,
            p.product_category_name,
            'unknown'
        ) AS product_category,

        f.seller_sk,
        s.seller_id,
        s.seller_city,
        s.seller_state,

        f.date_sk,
        d.calendar_date,
        d.calendar_year,
        d.calendar_quarter,
        d.calendar_month,
        d.month_name,
        d.year_month,
        d.week_of_year,
        d.day_name,
        d.is_weekend,

        f.price,
        f.freight_value,
        f.item_total_value

    FROM {catalog}.{schema}.gold_fact_order_items f

    INNER JOIN {catalog}.{schema}.secure_gold_dim_customer c
        ON f.customer_sk = c.customer_sk

    INNER JOIN {catalog}.{schema}.gold_dim_product p
        ON f.product_sk = p.product_sk

    INNER JOIN {catalog}.{schema}.gold_dim_seller s
        ON f.seller_sk = s.seller_sk

    INNER JOIN {catalog}.{schema}.gold_dim_date d
        ON f.date_sk = d.date_sk
""")

print("Secure dashboard view created")

### Validate dashboard source tables

**Purpose:** Check that the Gold and governed views required by the published dashboard exist and contain usable data.

**Inputs:** Gold fact, dimensions, and secure dashboard source views.

**Outputs:** Table existence, schema, and row-count evidence.

**Why it matters:** A refresh should not run when a dashboard source is missing or empty.

In [0]:
# Cell 4 — Preview the caller-specific dashboard rows after RLS and CLS are applied.
dashboard_df = spark.table(
    f"{catalog}.{schema}.secure_gold_order_items_dashboard"
)

print(
    "Dashboard-visible order items:",
    dashboard_df.count()
)

display(
    dashboard_df.select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "customer_id",
        "customer_state",
        "product_category",
        "seller_state",
        "price",
        "freight_value",
        "item_total_value"
    ).limit(10)
)

### Inspect dashboard-ready metrics

**Purpose:** Query the dashboard source at its intended analytical grain.

**Inputs:** `secure_gold_order_items_dashboard` and related Gold dimensions.

**Outputs:** KPI rows used for order volume, value, freight, and status visuals.

**Why it matters:** This confirms the dashboard model is readable and semantically complete before refresh.

In [0]:
# Cell 5 — Dynamically validate dashboard reconciliation, RLS, and CLS for any mapping.
dynamic_dashboard_validation_df = spark.sql(f"""
    WITH user_context AS (
        SELECT
            session_user() AS principal,
            is_account_group_member(
                'olist_pii_readers'
            ) AS is_pii_reader
    ),

    access_context AS (
        SELECT
            COALESCE(
                MAX(
                    CASE
                        WHEN a.customer_state = 'ALL'
                        THEN 1
                        ELSE 0
                    END
                ),
                0
            ) AS has_all_access,

            SORT_ARRAY(
                COLLECT_SET(a.customer_state)
            ) AS configured_states

        FROM {catalog}.{schema}.governance_customer_state_access a
        CROSS JOIN user_context u
        WHERE a.principal = u.principal
    ),

    expected AS (
        SELECT
            COUNT(*) AS expected_rows,
            COUNT(DISTINCT f.order_id) AS expected_orders,
            SUM(f.price) AS expected_total_price,
            SUM(f.freight_value) AS expected_total_freight,
            SUM(f.item_total_value) AS expected_total_value

        FROM {catalog}.{schema}.gold_fact_order_items f

        INNER JOIN {catalog}.{schema}.secure_gold_dim_customer c
            ON f.customer_sk = c.customer_sk
    ),

    dashboard_rows AS (
        SELECT
            d.*,
            u.is_pii_reader,
            a.has_all_access,
            a.configured_states

        FROM {catalog}.{schema}.secure_gold_order_items_dashboard d
        CROSS JOIN user_context u
        CROSS JOIN access_context a
    ),

    actual AS (
        SELECT
            COUNT(*) AS actual_rows,
            COUNT(DISTINCT order_id) AS actual_orders,
            SUM(price) AS actual_total_price,
            SUM(freight_value) AS actual_total_freight,
            SUM(item_total_value) AS actual_total_value,

            FIRST(is_pii_reader) AS is_pii_reader,
            FIRST(has_all_access) AS has_all_access,
            FIRST(configured_states) AS configured_states,

            SUM(
                CASE
                    WHEN has_all_access = 1
                         OR ARRAY_CONTAINS(
                             configured_states,
                             customer_state
                         )
                    THEN 0
                    ELSE 1
                END
            ) AS unauthorized_state_rows,

            SUM(
                CASE
                    WHEN is_pii_reader = FALSE
                         AND (
                             customer_id NOT LIKE '%...MASKED'
                             OR customer_unique_id NOT LIKE '%...MASKED'
                             OR customer_zip_code_prefix NOT LIKE '%**'
                         )
                    THEN 1

                    WHEN is_pii_reader = TRUE
                         AND (
                             customer_id LIKE '%...MASKED'
                             OR customer_unique_id LIKE '%...MASKED'
                             OR customer_zip_code_prefix LIKE '%**'
                         )
                    THEN 1

                    ELSE 0
                END
            ) AS cls_violations

        FROM dashboard_rows
    ),

    comparison AS (
        SELECT
            e.*,
            a.*,

            a.actual_rows
                - e.expected_rows AS row_difference,

            a.actual_orders
                - e.expected_orders AS order_difference,

            ROUND(
                a.actual_total_price
                - e.expected_total_price,
                2
            ) AS price_difference,

            ROUND(
                a.actual_total_freight
                - e.expected_total_freight,
                2
            ) AS freight_difference,

            ROUND(
                a.actual_total_value
                - e.expected_total_value,
                2
            ) AS value_difference

        FROM expected e
        CROSS JOIN actual a
    )

    SELECT
        *,
        CASE
            WHEN row_difference = 0
                 AND order_difference = 0
                 AND price_difference = 0
                 AND freight_difference = 0
                 AND value_difference = 0
                 AND unauthorized_state_rows = 0
                 AND cls_violations = 0
            THEN 'PASS'
            ELSE 'FAIL'
        END AS status
    FROM comparison
""")

display(dynamic_dashboard_validation_df)

### Validate KPI coverage

**Purpose:** Check the key dashboard measures and dimensions used by the published layout.

**Inputs:** Dashboard source view, Gold fact metrics, and status categories.

**Outputs:** Validation results for expected KPI columns and values.

**Why it matters:** KPI coverage prevents a technically successful refresh from publishing an incomplete business view.

In [0]:
display(
    spark.sql("""
        SELECT
            session_user() AS current_user,

            is_account_group_member(
                'olist_pii_readers'
            ) AS is_pii_reader
    """)
)

### Record dashboard validation status

**Purpose:** Persist or display the final dashboard-readiness result.

**Inputs:** Results from source existence, policy, and KPI checks.

**Outputs:** A validation outcome consumed by Job orchestration.

**Why it matters:** The dashboard refresh must follow successful validation rather than merely follow data availability.

In [0]:
display(
    spark.table(
        f"{catalog}.{schema}.secure_gold_order_items_dashboard"
    )
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_state"
    )
    .limit(10)
)

### Finish dashboard readiness checks

**Purpose:** Complete the final display or assertion for dashboard publication readiness.

**Inputs:** The validated dashboard source and expected development metrics.

**Outputs:** A clear success signal for the downstream refresh dependency.

**Why it matters:** This notebook is the explicit quality boundary before dashboard consumption.